# Penguin-VL Inference Recipes

This notebook mirrors the cases in `inference/example_penguinvl.py` and presents them in a notebook-friendly format that is easier to read on GitHub after execution.

## What This Notebook Covers

1. Generate Python code from a screenshot of an algorithm problem.
2. Parse a newspaper page with OCR-style prompting.
3. Produce creative writing from an artwork.
4. Convert a long table image into Markdown.
5. Run a multi-round chart-understanding conversation.
6. Run a multi-round video-understanding conversation.
7. Mix video and image inputs in a single prompt.
8. Compare with a plain-text-only prompt.

## Before You Run It

- Use the same Python environment that you use for `python inference/example_penguinvl.py`.
- `transformers==4.51.3` is recommended by the repo.
- Set `MODEL_PATH` below to a Hugging Face model ID or a local checkpoint.
- After running the notebook, save it so GitHub can render the outputs inline.


## General: Import Libraries and Locate the Repo

This cell keeps the notebook portable. It searches upward from the current working directory until it finds `inference/example_penguinvl.py`.


In [ ]:
import os
import html
import base64
import mimetypes
from pathlib import Path

os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

import torch
import markdown2
from IPython.display import HTML, Markdown, display
from transformers import AutoModelForCausalLM, AutoProcessor


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'inference' / 'example_penguinvl.py').exists():
            return path
    raise FileNotFoundError('Could not locate the Penguin-VL repo root from the current working directory.')


REPO_ROOT = find_repo_root(Path.cwd())
ASSETS_DIR = REPO_ROOT / 'assets' / 'inputs'
SCRIPT_PATH = REPO_ROOT / 'inference' / 'example_penguinvl.py'

print('Located Penguin-VL repo assets successfully.')
print('Script target: inference/example_penguinvl.py')
print('Assets directory: assets/inputs')


## General: Load Model and Processor

This cell intentionally matches the loading logic in `inference/example_penguinvl.py` so the notebook stays aligned with the script.


In [ ]:
MODEL_PATH = os.environ.get('PENGUIN_VL_MODEL_PATH', 'tencent/Penguin-VL-8B')
MODEL_LABEL = Path(MODEL_PATH).name if Path(MODEL_PATH).is_absolute() else MODEL_PATH
print('Using MODEL_PATH:', MODEL_LABEL)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    device_map={'': 'cuda:0'},
    torch_dtype=torch.bfloat16,
    attn_implementation='flash_attention_2',
)
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)


## General: Notebook Helpers

The helper functions below replicate the current `infer()` behavior from the Python script and render question/answer outputs in a cleaner layout.


In [ ]:
NOTEBOOK_STYLE = """
<style>
.penguin-panel {
    border: 1px solid #dbe3ef;
    border-radius: 18px;
    padding: 20px 24px;
    margin: 14px 0 20px;
    background: #ffffff;
    box-shadow: 0 1px 2px rgba(15, 23, 42, 0.05);
}
.penguin-grid {
    display: grid;
    grid-template-columns: repeat(2, minmax(0, 1fr));
    gap: 18px;
    margin: 14px 0 28px;
}
.penguin-label {
    color: #445fd1;
    text-transform: uppercase;
    letter-spacing: 0.08em;
    font-size: 0.78rem;
    font-weight: 700;
    margin-bottom: 10px;
}
.penguin-title {
    font-size: 1.35rem;
    font-weight: 700;
    color: #1f2a44;
    margin-bottom: 8px;
}
.penguin-body {
    color: #2a3652;
    line-height: 1.7;
    font-size: 1rem;
    white-space: pre-wrap;
}
.penguin-caption {
    color: #5e6b84;
    font-size: 0.95rem;
    margin-top: 12px;
}
.penguin-table-wrap {
    overflow-x: auto;
    margin-top: 2px;
}
.penguin-markdown table {
    width: 100%;
    border-collapse: collapse;
    font-size: 0.95rem;
}
.penguin-markdown th,
.penguin-markdown td {
    border: 1px solid #dbe3ef;
    padding: 10px 12px;
    text-align: left;
    vertical-align: top;
}
.penguin-markdown thead th {
    background: #eef3ff;
}
.penguin-markdown tbody tr:nth-child(even) {
    background: #f8fbff;
}
.penguin-markdown p {
    margin: 0;
}
.penguin-markdown p + p {
    margin-top: 12px;
}
.penguin-markdown h1,
.penguin-markdown h2,
.penguin-markdown h3,
.penguin-markdown h4 {
    color: #1f2a44;
    line-height: 1.3;
    margin: 0 0 12px;
}
.penguin-markdown ul,
.penguin-markdown ol {
    margin: 8px 0 0;
    padding-left: 22px;
}
.penguin-markdown li + li {
    margin-top: 6px;
}
.penguin-markdown pre {
    margin: 12px 0 0;
    padding: 16px 18px;
    overflow-x: auto;
    border-radius: 14px;
    background: #f4f7fb;
    border: 1px solid #dbe3ef;
    line-height: 1.6;
}
.penguin-markdown code {
    font-family: ui-monospace, SFMono-Regular, SFMono-Regular, Menlo, Monaco, Consolas, Liberation Mono, Courier New, monospace;
    font-size: 0.92em;
}
.penguin-markdown :not(pre) > code {
    background: #eef3ff;
    padding: 0.15rem 0.35rem;
    border-radius: 6px;
}
.penguin-markdown pre code {
    background: transparent;
    padding: 0;
}
.penguin-image {
    width: 100%;
    border-radius: 12px;
}
@media (max-width: 900px) {
    .penguin-grid {
        grid-template-columns: 1fr;
    }
}
</style>
"""

display(HTML(NOTEBOOK_STYLE))


def infer(conversation, max_new_tokens=2048, temperature=0.1, do_sample=True):
    inputs = processor(
        conversation=conversation,
        add_system_prompt=True,
        add_generation_prompt=True,
        return_tensors='pt',
    )
    inputs = {k: v.cuda() if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
    if 'pixel_values' in inputs:
        inputs['pixel_values'] = inputs['pixel_values'].to(torch.bfloat16)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=do_sample,
    )

    if output_ids.shape[1] > inputs['input_ids'].shape[1]:
        response_ids = output_ids[:, inputs['input_ids'].shape[1]:]
    else:
        response_ids = output_ids

    response = processor.batch_decode(response_ids, skip_special_tokens=True)[0].strip()
    return response


def _html_breaks(text: str) -> str:
    return html.escape(text).replace('\\n', '<br>')


def _image_to_data_uri(image_path: Path) -> str:
    mime = mimetypes.guess_type(str(image_path))[0] or 'image/png'
    with open(image_path, 'rb') as f:
        encoded = base64.b64encode(f.read()).decode('ascii')
    return f'data:{mime};base64,{encoded}'


def _public_path(path: Path) -> str:
    try:
        return str(path.relative_to(REPO_ROOT))
    except ValueError:
        return path.name


def _looks_like_markdown_table(text: str) -> bool:
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    if len(lines) < 2:
        return False
    separator = lines[1].replace('|', '').replace(':', '').replace('-', '').strip()
    return lines[0].count('|') >= 2 and separator == ''


def _looks_like_rich_markdown(text: str) -> bool:
    stripped = text.strip()
    if not stripped:
        return False
    if '```' in stripped:
        return True
    if _looks_like_markdown_table(stripped):
        return True
    markdown_prefixes = ('# ', '## ', '### ', '#### ', '- ', '* ', '> ')
    if any(line.lstrip().startswith(markdown_prefixes) for line in stripped.splitlines()):
        return True
    return any(line.lstrip().startswith(f'{i}. ') for line in stripped.splitlines() for i in range(1, 10))


def _render_answer_html(answer: str) -> str:
    if _looks_like_rich_markdown(answer):
        rendered_html = markdown2.markdown(answer, extras=['fenced-code-blocks', 'tables', 'break-on-newline', 'code-friendly'])
        if _looks_like_markdown_table(answer):
            return f"<div class='penguin-body penguin-markdown'><div class='penguin-table-wrap'>{rendered_html}</div></div>"
        return f"<div class='penguin-body penguin-markdown'>{rendered_html}</div>"
    return f"<div class='penguin-body'>{_html_breaks(answer)}</div>"


def show_image_panel(image_path: Path, title='Input Image', caption=None):
    caption_html = ''
    if caption:
        caption_html = f"<div class='penguin-caption'>{_html_breaks(caption)}</div>"
    display(HTML(
        f"""
        <div class='penguin-panel'>
            <div class='penguin-label'>{html.escape(title)}</div>
            <img class='penguin-image' src='{_image_to_data_uri(image_path)}' />
            {caption_html}
        </div>
        """
    ))


def show_video_panel(video_path: Path, fps=1, max_frames=180):
    public_path = _public_path(video_path)
    display(HTML(
        f"""
        <div class='penguin-panel'>
            <div class='penguin-label'>Input Video</div>
            <div class='penguin-title'>{html.escape(video_path.name)}</div>
            <div class='penguin-body'><strong>Source:</strong> {html.escape(public_path)}<br><strong>FPS:</strong> {fps}<br><strong>Max Frames:</strong> {max_frames}</div>
            <div class='penguin-caption'>The notebook keeps video display lightweight for GitHub. Open the local file if you want to inspect the raw video while reviewing the saved outputs.</div>
        </div>
        """
    ))


def show_turn(question: str, answer: str, round_id=None):
    response_label = 'Model Response' if round_id is None else f'Round {round_id} Response'
    answer_html = _render_answer_html(answer)
    display(HTML(
        f"""
        <div class='penguin-grid'>
            <div class='penguin-panel'>
                <div class='penguin-label'>Question</div>
                <div class='penguin-body'>{_html_breaks(question)}</div>
            </div>
            <div class='penguin-panel'>
                <div class='penguin-label'>{html.escape(response_label)}</div>
                {answer_html}
            </div>
        </div>
        """
    ))


def run_single_image_case(title: str, image_name: str, question: str):
    image_path = ASSETS_DIR / image_name
    display(Markdown(f'### {title}'))
    show_image_panel(image_path, caption=image_name)

    conversation = [
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': {'image_path': str(image_path)}},
                {'type': 'text', 'text': question},
            ]
        }
    ]
    response = infer(conversation)
    show_turn(question, response)
    return response


def run_multiround_image_case(title: str, image_name: str, questions):
    image_path = ASSETS_DIR / image_name
    display(Markdown(f'### {title}'))
    show_image_panel(image_path, caption=image_name)

    conversation = []
    answers = []
    for round_id, question in enumerate(questions, start=1):
        if round_id == 1:
            conversation.append(
                {
                    'role': 'user',
                    'content': [
                        {'type': 'image', 'image': {'image_path': str(image_path)}},
                        {'type': 'text', 'text': question},
                    ]
                }
            )
        else:
            conversation.append({'role': 'user', 'content': question})

        response = infer(conversation)
        conversation.append({'role': 'assistant', 'content': response})
        show_turn(question, response, round_id=round_id)
        answers.append(response)

    return answers


def run_multiround_video_case(title: str, video_name: str, questions, fps=1, max_frames=180):
    video_path = ASSETS_DIR / video_name
    display(Markdown(f'### {title}'))
    show_video_panel(video_path, fps=fps, max_frames=max_frames)

    conversation = [{'role': 'system', 'content': 'You are a helpful assistant.'}]
    answers = []
    for round_id, question in enumerate(questions, start=1):
        if round_id == 1:
            conversation.append(
                {
                    'role': 'user',
                    'content': [
                        {'type': 'video', 'video': {'video_path': str(video_path), 'fps': fps, 'max_frames': max_frames}},
                        {'type': 'text', 'text': question},
                    ]
                }
            )
        else:
            conversation.append({'role': 'user', 'content': question})

        response = infer(conversation)
        conversation.append({'role': 'assistant', 'content': response})
        show_turn(question, response, round_id=round_id)
        answers.append(response)

    return answers


def run_mixed_case(title: str, video_name: str, image_name: str, question: str, fps=1, max_frames=180):
    video_path = ASSETS_DIR / video_name
    image_path = ASSETS_DIR / image_name
    display(Markdown(f'### {title}'))
    show_video_panel(video_path, fps=fps, max_frames=max_frames)
    show_image_panel(image_path, title='Reference Image', caption=image_name)

    conversation = [
        {
            'role': 'user',
            'content': [
                {'type': 'text', 'text': question},
                {'type': 'video', 'video': {'video_path': str(video_path), 'fps': fps, 'max_frames': max_frames}},
                {'type': 'text', 'text': '\\n\\nImage\\n'},
                {'type': 'image', 'image': {'image_path': str(image_path)}},
            ]
        }
    ]
    response = infer(conversation)
    show_turn(question, response)
    return response


def run_text_case(title: str, question: str):
    display(Markdown(f'### {title}'))
    conversation = [{'role': 'user', 'content': question}]
    response = infer(conversation)
    show_turn(question, response)
    return response


## Section 1: Single Image Understanding

This section mirrors the single-image examples in the Python script. Each case uses one image plus one user prompt.


In [ ]:
leetcode_response = run_single_image_case(
    title='Example 1: Code Generation from an Algorithm Screenshot',
    image_name='leetcode.png',
    question='please think this problem step by step and give the python code solution',
)


In [ ]:
ocr_response = run_single_image_case(
    title='Example 2: OCR-Style Parsing from a Newspaper Page',
    image_name='newspaper.png',
    question='please parse the text content in the paragraphs, from left to right, from top to bottom.',
)


In [ ]:
poem_response = run_single_image_case(
    title='Example 3: Creative Writing from an Artwork',
    image_name='horse_poet.png',
    question='Write a short poem inspired by this image. Capture the relationship between the man and the horse, as well as the traditional, historical atmosphere of the painting.',
)


In [ ]:
table_response = run_single_image_case(
    title='Example 4: Long Table Parsing to Markdown',
    image_name='2b_table_result.png',
    question='please output the table content in markdown format.',
)


## Section 2: Multi-Round Chart Understanding

This case demonstrates how to carry conversation history forward. The image is only attached in round 1; later rounds rely on the stored assistant context.


In [ ]:
chart_answers = run_multiround_image_case(
    title='Example 5: Multi-Round Chart Understanding',
    image_name='chart_understanding.png',
    questions=[
        "Look at the 'Nonmetropolitan' line. In what approximate year does it reach its absolute lowest point on the chart, and what is the approximate percent change at that time?",
        'Compare the three lines over the entire 50-year period. Which demographic category exhibits the highest volatility (the most dramatic peaks and valleys), and which one appears to be the most stable?',
    ],
)


## Section 3: Multi-Round Video Understanding

This case mirrors the multi-round video example in the script. The notebook records the prompt sequence and each generated answer while keeping the video display lightweight for GitHub.


In [ ]:
video_answers = run_multiround_video_case(
    title='Example 6: Multi-Round Video Understanding',
    video_name='video-example.mp4',
    questions=[
        'please describe the video in details',
        'At what timestamps is the Summar Palace mentioned?',
        'At what timestamps is the CHANG AN AVENUE mentioned?',
        'At what timestamps is the THE FINANCIAL STREET FORUM mentioned?',
    ],
    fps=1,
    max_frames=180,
)


## Section 4: Mixed-Modal Prompting

This example combines a video and an image in one user turn, then asks the model to produce a single creative response.


In [ ]:
mixed_response = run_mixed_case(
        title='Example 7: Mixed Video + Image Story Generation',
        video_name='polar_bear.mp4',
        image_name='horse_poet.png',
        question='Write a fairy tale based on the video and the image below:\\nVideo\\n',
        fps=1,
        max_frames=180,
    )


## Section 5: Text-Only Baseline

A plain-text example is useful as a minimal baseline when you want to compare multimodal behavior against a standard chat prompt.


In [ ]:
text_response = run_text_case(
    title='Example 8: Plain Text Conversation',
    question='There are ten birds in a tree. If you shoot and kill one, how many are left?',
)


## Optional Appendix: Capture the Raw Script Output

The cells above are the notebook-friendly version of the examples. If you also want exact terminal parity with `python3 inference/example_penguinvl.py`, run the next cell. It will launch the script in a subprocess and print the raw stdout and stderr.

Note: this loads the model again, so it is slower and uses additional GPU memory while it runs.


In [ ]:
RUN_RAW_SCRIPT_APPENDIX = False

if RUN_RAW_SCRIPT_APPENDIX:
    import subprocess

    cmd = [
        'python3',
        'inference/example_penguinvl.py',
        '--model-path',
        MODEL_PATH,
    ]

    result = subprocess.run(
        cmd,
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        check=False,
    )

    print('Return code:', result.returncode)
    print('\n=== STDOUT ===\n')
    print(result.stdout)
    print('\n=== STDERR ===\n')
    print(result.stderr)
else:
    print('Skipping raw script appendix. Set RUN_RAW_SCRIPT_APPENDIX = True to enable it.')


## Optional: How to Execute This Notebook from the Terminal

From the repo root, you can either open it interactively:

```bash
jupyter lab
```

or execute it in-place so the outputs are saved for GitHub rendering:

```bash
jupyter nbconvert --to notebook --execute --inplace inference/notebooks/01_penguinvl_inference_recipes.ipynb
```

If you use a custom environment or launcher, make sure the notebook kernel points to the same environment that successfully runs `python inference/example_penguinvl.py`.
